# Liputan6 Translation (Indonesian -> English)

Produces `translate/{id}.txt` (English) for every sentence. The AMR parser needs
these because the model is a **concat** model (`id_ID <indo> en_XX <eng>`).

## Setup
1. **Add Input:** the `liputan6-data` dataset (contains `analysis_data.csv`).
2. **Accelerator:** GPU **T4**.
3. **Internet: ON** (needed to download NLLB-1.3B).
4. **Persistence: Files only** (optional convenience for interactive runs).

## How to run - use Save & Run All, do NOT babysit it
Click **Save Version -> Save & Run All (Commit)**, then close the browser.
Kaggle runs the whole notebook headless and **saves the output automatically**
when it finishes. You do *not* click Save Version at the end, and you do *not*
stop it midway.

(The interactive Run buttons are fine for a quick test, but anything in
`/kaggle/working` is lost when an interactive session ends unless you save it.)

## Why it stops early on purpose
`MAX_NEW` and `TIME_BUDGET_H` end each run cleanly, comfortably inside Kaggle's
~12h cap. A run that hits the wall can fail **without saving anything**, so each
run deliberately finishes early and saves. Repeat until 100%.

## Chaining runs (195,779 sentences won't fit in one run)
- **Run 1:** set `PREV_DIR = ""`. When it finishes: Output panel -> **New Dataset**,
  name it **`liputan6-translate`**.
- **Run 2+:** leave `PREV_DIR` as-is (it already points at that dataset). When it
  finishes, update `liputan6-translate` with a **new version**.

Already-translated IDs are skipped, so no work is ever repeated.

> **Tip:** on your very first run set `MAX_NEW = 2000`. It finishes in minutes,
> proves the whole chain works, and tells you the real rate so you can size
> `MAX_NEW` properly.

In [ ]:
import os, shutil, torch, pandas as pd
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

# ============================================================
# KAGGLE PATHS - all stable paths are explicit and independent.
# ============================================================
CSV_PATH   = "/kaggle/input/datasets/fedrianzdharma/liputan6-data/analysis_data.csv"
OUTPUT_DIR = "/kaggle/working/translate"

# CROSS-SESSION RESUME:
# After a run finishes, save its output as a DATASET named "liputan6-translate"
# (Output panel -> New Dataset). Set PREV_DIR = "" on the FIRST run.
PREV_DIR   = "/kaggle/input/datasets/fedrianzdharma/liputan6-translate/translate"
# PREV_DIR = ""   # <- use this on the very first run

# ============================================================
# RUN BUDGET - keep each Save & Run All (Commit) comfortably under
# Kaggle's ~12h limit so the run FINISHES and its output is saved.
# A run that hits the wall can fail without saving anything.
#   MAX_NEW      : stop after this many NEW translations this run
#   TIME_BUDGET_H: also stop after this many hours, whichever comes first
# Set MAX_NEW = None for no cap (only for a final short run).
# ============================================================
MAX_NEW       = 40000
TIME_BUDGET_H = 10.5

os.makedirs(OUTPUT_DIR, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device      :", device)
print("CSV exists  :", os.path.exists(CSV_PATH), "|", CSV_PATH)
print("PREV_DIR    :", os.path.isdir(PREV_DIR) if PREV_DIR else False, "|", PREV_DIR)

## Load NLLB model & tokenizer

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

MODEL_NAME = "facebook/nllb-200-distilled-1.3B"
# src_lang MUST be set so Indonesian is tokenized/flagged correctly
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, src_lang="ind_Latn")
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(device)
model.eval()
eng_id = tokenizer.convert_tokens_to_ids("eng_Latn")
print("NLLB loaded. eng_Latn id =", eng_id)

## Dataset & DataLoader

In [ ]:
class TextDataset(Dataset):
    def __init__(self, csv_path):
        self.df = pd.read_csv(csv_path, dtype={"id": str})
        print("Total rows:", len(self.df))
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        r = self.df.iloc[idx]
        return {"id": r["id"], "text": str(r["text"])}

def collate(batch):
    return [b["id"] for b in batch], [b["text"] for b in batch]

ds = TextDataset(CSV_PATH)
# batch_size 4 keeps the 1.3B model within T4 (16GB) memory at max_length=1024
loader = DataLoader(ds, batch_size=4, collate_fn=collate)

## Translate (resumable, bug-fixed: saves the ENGLISH output)

In [ ]:
# Build the set of IDs already translated, from BOTH this run's output dir
# and the previous run's mounted output. This is what makes the job
# resumable ACROSS sessions (/kaggle/working starts empty every session).
def done_ids():
    ids = set()
    for d in [PREV_DIR, OUTPUT_DIR]:
        if d and os.path.isdir(d):
            for fn in os.listdir(d):
                if fn.endswith(".txt"):
                    ids.add(fn[:-4])
    return ids

DONE = done_ids()
print(f"Already translated (carried over): {len(DONE)}")

In [ ]:
import time
start_t = time.time()
budget_s = TIME_BUDGET_H * 3600
done, skipped, stop_reason = 0, 0, "completed all remaining"

pbar = tqdm(loader, desc="Translating")
for ids, texts in pbar:
    # --- stop conditions: end the run CLEANLY so the output gets saved ---
    if MAX_NEW is not None and done >= MAX_NEW:
        stop_reason = f"hit MAX_NEW={MAX_NEW}"
        break
    if time.time() - start_t > budget_s:
        stop_reason = f"hit TIME_BUDGET_H={TIME_BUDGET_H}"
        break

    keep_ids, keep_texts = [], []
    for i, t in zip(ids, texts):
        if i in DONE:
            skipped += 1
            continue
        keep_ids.append(i)
        keep_texts.append(t)
    if not keep_ids:
        continue

    enc = tokenizer(keep_texts, return_tensors="pt", padding=True,
                    truncation=True, max_length=1024).to(device)
    with torch.no_grad():
        gen = model.generate(**enc, forced_bos_token_id=eng_id,
                             max_length=1024, num_beams=1)
    outs = tokenizer.batch_decode(gen, skip_special_tokens=True)
    for i, o in zip(keep_ids, outs):
        with open(os.path.join(OUTPUT_DIR, f"{i}.txt"), "w", encoding="utf-8") as f:
            f.write(o)          # saves the TRANSLATION (original bug saved Indonesian)
        DONE.add(i)
    done += len(keep_ids)
    pbar.set_postfix(new=done, elapsed_h=f"{(time.time()-start_t)/3600:.2f}")

pbar.close()
print(f"Stopped because : {stop_reason}")
print(f"Newly translated: {done} | skipped (already done): {skipped}")
print(f"Elapsed         : {(time.time()-start_t)/3600:.2f} h")

## Carry forward previous output (keeps each version cumulative)

In [ ]:
# Make THIS version's output cumulative and self-contained: copy forward every
# file from the previous run that isn't already here. The next session then only
# needs to attach THIS version, not the whole chain.
copied = 0
if PREV_DIR and os.path.isdir(PREV_DIR):
    have = set(os.listdir(OUTPUT_DIR))
    for fn in tqdm(os.listdir(PREV_DIR), desc="Carrying forward"):
        if fn.endswith(".txt") and fn not in have:
            shutil.copyfile(os.path.join(PREV_DIR, fn), os.path.join(OUTPUT_DIR, fn))
            copied += 1

total = len([f for f in os.listdir(OUTPUT_DIR) if f.endswith('.txt')])
import pandas as _pd
expected = len(_pd.read_csv(CSV_PATH, dtype={'id': str}))
print(f"Carried forward : {copied}")
print(f"TOTAL in output : {total} / {expected}  ({total/max(expected,1)*100:.1f}%)")
print("\n>>> Click 'Save Version' NOW or this output is lost when the session ends.")
print(">>> Next session: attach THIS version's output and set PREV_DIR to it.")